<a href="https://colab.research.google.com/github/Diamanth/prueba-tecnica-techstore/blob/main/PruebaTecnicaEduardoRuiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# ==========================================
# CONFIGURACIÓN DEL ENTORNO (EJECUTAR PRIMERO)
# ==========================================
import pandas as pd
import sqlite3

# 1) URLs de archivos crudos (Raw) en GitHub
urls = {
    'clientes': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/clientes.csv",
    'ventas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/ventas.csv",
    'trabajadores': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/trabajadores.csv",
    'cargos': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/cargos.csv",
    'areas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/areas.csv"
}

print("Preparando base de datos 'techstore.db'.. .")

# 2) Crear la conexión y la base de datos
conn = sqlite3.connect('techstore.db')

# 3) Leer cada CSV e insertarlo como tabla en la base de datos
for tabla, url in urls.items():
    try:
        df = pd.read_csv(url, sep=None, engine='python')
        df.to_sql(tabla, conn, index=False, if_exists='replace')
        print(f"Tabla '{tabla}' cargada exitosamente.")
    except Exception as e:
        print(f"Error al cargar la tabla '{tabla}': {e}")

conn.close()
print("¡Entorno listo! La base de datos relacional está disponible para ser consultada.")

Preparando base de datos 'techstore.db'.. .
Tabla 'clientes' cargada exitosamente.
Tabla 'ventas' cargada exitosamente.
Tabla 'trabajadores' cargada exitosamente.
Tabla 'cargos' cargada exitosamente.
Tabla 'areas' cargada exitosamente.
¡Entorno listo! La base de datos relacional está disponible para ser consultada.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Contexto de la Prueba: E-Commerce "TechStore"**

*Escenario: Eres el nuevo analista de datos de TechStore. El equipo ha notado una fluctuación en los ingresos durante el último trimestre y necesita entender qué está pasando.*

**Instrucciones Iniciales:**

1.   La prueba tiene una duración estimada de 45 mins, pero tendrás 72 horas para enviar tus resultados.
2.	Luego de terminar tus ejercicios envíame el enlace en compartir para tu entorno donde fue desarrollado.
3.	Cualquier archivo externo generado debes subirlo y compartir el enlace en una nueva hoja de google colab, en caso de usar hojas de cálculo de Google debes copiar el enlace en otra hoja de google colab, recuerda titular dicho enlace para saber que iré a mirar.
4.	Dentro de la prueba, ve a Archivo > Guardar una copia en Drive para tener tu propia versión editable de esta prueba.
5.	Ejecuta la primera celda de este cuaderno. Esto generará automáticamente una base de datos SQLite llamada techstore.db en este entorno.
6.	El diccionario de datos disponible en la base de datos es:
*   clientes: id, rut, nombre, segundo_nombre, apellido, segundo_apellido, edad, fecha_nacimiento
*   ventas: id, id_cliente, id_vendedor, dte, total_dte, estado, descripcion
*   trabajadores: id, id_cargo, nombre, apellido, rut
*   cargos: id, nombre_cargo, descripcion, id_area
*   areas: id, nombre_area, descripcion

*disclaimer: Puedes usar IA, sin embargo, revisaré cada linea de código para validar la efectividad y también la totalidad del cumplimiento de las respuestas.*

In [7]:
# @title
import sqlite3
import pandas as pd

# Forma nativa de tablas en Colab
%load_ext google.colab.data_table
conn = sqlite3.connect('techstore.db')
tabla_a_revisar = 'ventas'
query = f"SELECT * FROM {tabla_a_revisar}"
df = pd.read_sql_query(query, conn)
# Exporto a excel para poder hacer un analisis de datos
df.to_excel('reporte_ventas_techstore.xlsx', index=False)
conn.close()
df

The google.colab.data_table extension is already loaded. To reload it, use:
  %reload_ext google.colab.data_table


,id,id_cliente,id_vendedor,dte,total_dte,estado,descripcion
0,1,191,15,10001,650631,Entregado,"2x Disco Duro Externo 2TB, 3x Mouse Ergonómico..."
1,2,109,3,10004,903198,Entregado,"2x Escáner de Documentos Portátil, 1x Mouse Er..."
2,3,76,10,10007,231213,Entregado,"2x Escáner de Documentos Portátil, 2x Memoria ..."
3,4,158,14,10009,752982,Sin entregar,"3x Licencia Software Antivirus 1 Año, 3x Lapto..."
4,5,132,6,10011,36388,Entregado,3x Mouse Ergonómico Inalámbrico
...,...,...,...,...,...,...,...
1195,1196,117,12,12408,251364,Entregado,"1x Teclado Mecánico RGB, 2x Licencia Software ..."
1196,1197,138,8,12409,64199,Sin entregar,"1x Impresora Láser Multifuncional, 1x Licencia..."
1197,1198,149,4,12412,84752,Entregado,3x Silla de Escritorio Ergonómica
1198,1199,28,5,12413,487762,Entregado,"1x Impresora Láser Multifuncional, 2x Mouse Er..."


**Parte 1: Conexión y Extracción (SQL & Python)**

*Conéctate a la base de datos techstore.db usando Python (por ejemplo, con la librería sqlite3 o sqlalchemy). Luego, resuelve las siguientes consultas ejecutando SQL puro dentro de tu código Python:*
1.   **Ventas Totales:** Escribe una consulta para calcular el ingreso total generado por órdenes con estado "Entregado" y el % del total de ventas que representa.
2.   **Top Clientes:** Encuentra los 5 usuarios que han gastado más dinero históricamente, mostrando su id, rut y el total_dte gastado.
3.   **Retención:** Calcula el porcentaje de usuarios que realizaron una segunda compra dentro de los 200 dte siguientes (si alguien compró el dte 11721 y compró nuevamente antes ó justamente en el dte 11921 es considerado).

In [ ]:
# @title
import sqlite3
import pandas as pd

conn = sqlite3.connect('techstore.db')

# Ventas totales
query = """SELECT SUM(CASE WHEN estado = 'Entregado' THEN total_dte ELSE 0 END)
        AS ingreso_entregado, SUM(total_dte) AS ingreso_total, ROUND(SUM(CASE
        WHEN estado = 'Entregado' THEN total_dte ELSE 0 END)
        * 100.0 / SUM(total_dte), 2) AS porcentaje FROM ventas"""

ventas = pd.read_sql_query(query, conn)

print("Ventas totales")
print(ventas)

# Top clientes
query = """SELECT c.id, c.rut, SUM(v.total_dte) AS total_gastado FROM clientes c
        JOIN ventas v ON c.id = v.id_cliente WHERE v.estado = 'Entregado'
        GROUP BY c.id, c.rut ORDER BY total_gastado DESC LIMIT 5"""

clientes = pd.read_sql_query(query, conn)

print("\nTop 5 clientes (Ventas Entregadas)")
print(clientes)

# Retencion
query = """WITH compras_numeradas AS (SELECT id_cliente, dte,ROW_NUMBER() OVER(
        PARTITION BY id_cliente ORDER BY dte) AS num_compra FROM ventas
        WHERE estado = 'Entregado'), primera_y_segunda AS (SELECT c1.id_cliente,
        c1.dte AS dte_primera, c2.dte AS dte_segunda FROM compras_numeradas c1
        LEFT JOIN compras_numeradas c2 ON c1.id_cliente = c2.id_cliente
        AND c2.num_compra = 2 WHERE c1.num_compra = 1)
        SELECT COUNT(*) AS total_clientes, COUNT(CASE WHEN dte_segunda
        <= dte_primera + 200 THEN 1 END) AS clientes_retenidos, ROUND(
        COUNT(CASE WHEN dte_segunda <= dte_primera + 200 THEN 1 END)
        * 100.0 / COUNT(*), 2) AS porcentaje_retencion FROM primera_y_segunda;"""

retencion = pd.read_sql_query(query, conn)

print("Retención")
print(retencion)

conn.close()

Ventas totales
   ingreso_entregado  ingreso_total  porcentaje
0          477191018      588673484       81.06

Top 5 clientes (Ventas Entregadas)
    id         rut  total_gastado
0  145  14605016-6        5604037
1   82  21866669-8        5466055
2   76  16189034-0        5086756
3   81  10212267-4        5012359
4   73  15871360-8        4858617
Retención
   total_clientes  clientes_retenidos  porcentaje_retencion
0             244                  67                 27.46


**Parte 2: Análisis Exploratorio (Python)**

*Trae las tablas necesarias a DataFrames de Pandas y resuelve:*

1.	**Limpieza:** Identifica y muestra los valores nulos o duplicados en la tabla de clientes. Comenta tu criterio.
2.	**Métricas:** Identifica el top 10 de total_dte más altos, el top 3 de clientes por volumen de transacciones (cantidad de compras), y el top 3 de vendedores (id_vendedor) con mayor monto acumulado, solo considera vendedores, si hay otros cargos comerciales que han realizado ventas debes ignorarlos.
3.	**Volumen:** Crea un DataFrame del top 10 de productos mas vendidos ordenado de mayor a menor que muestre el id de la venta, nombre del producto y unidades vendidas (deberás procesar la columna descripcion).


In [ ]:
# @title
import sqlite3
import pandas as pd
# 1. Limpieza: Parte uno de Análisis Exploratorio (Python)

conn = sqlite3.connect('techstore.db')
clientes = pd.read_sql_query("SELECT * FROM clientes", conn)
conn.close()

# Nulos
print("\nValores nulos por columna:")
print(clientes.isnull().sum())

# Duplicados
duplicados_id = clientes[clientes.duplicated(subset='id', keep=False)]
duplicados_rut = clientes[clientes.duplicated(subset='rut', keep=False)]
duplicados_completos = clientes[clientes.duplicated(keep=False)]

print(f"\nFilas duplicadas (registro completo): {duplicados_completos.shape[0]}")
print(f"RUTs repetidos: {duplicados_rut.shape[0]}")
print(f"IDs repetidos: {duplicados_id.shape[0]}")

if not duplicados_rut.empty:
    print("\nClientes con RUT duplicado:")
    print(duplicados_rut[['id', 'rut', 'nombre', 'apellido']].to_string(index=False))


Valores nulos por columna:
id                  0
rut                 0
nombre              0
segundo_nombre      0
apellido            0
segundo_apellido    0
edad                4
fecha_nacimiento    0
dtype: int64

Filas duplicadas (registro completo): 0
RUTs repetidos: 4
IDs repetidos: 0

Clientes con RUT duplicado:
 id        rut  nombre apellido
  4 10352896-8 Javiera    Silva
104 12626181-0   Diego     Soto
199 10352896-8 Javiera    Silva
251 12626181-0   Diego     Soto


In [ ]:
# @title
import sqlite3
import pandas as pd
# 2. Métricas: Parte dos de Análisis Exploratorio (Python)

conn = sqlite3.connect('techstore.db')

# Top 10 ventas más altas
q_top10 = """
SELECT id, id_cliente, id_vendedor, dte, total_dte, estado
FROM ventas
ORDER BY total_dte DESC
LIMIT 10;
"""
top10 = pd.read_sql_query(q_top10, conn)
print("TOP 10 · Ventas más altas (total_dte)")
print(top10.to_string(index=False))

# Top 3 clientes por volumen de transacciones

q_top_clientes_freq = """ SELECT c.id AS id_cliente, c.rut, c.nombre, c.apellido,
                      COUNT(v.id) AS cantidad_compras FROM ventas v JOIN clientes c
                      ON v.id_cliente = c.id GROUP BY c.id, c.rut, c.nombre, c.apellido
                      ORDER BY cantidad_compras DESC LIMIT 3;"""

top_clientes_freq = pd.read_sql_query (q_top_clientes_freq, conn)

print("\nTOP 3 · Clientes por volumen de transacciones")
print(top_clientes_freq.to_string(index=False))

# Top 3 vendedores por monto acumulado
# Solo se consideran cargos de vendedores: Ejecutivo de Ventas Senior y Junior
q_top_vendedores = """SELECT t.id AS id_vendedor, t.nombre || ' ' || t.apellido
                  AS nombre_vendedor, c.nombre_cargo, SUM(v.total_dte) AS monto_acumulado,
                  COUNT(v.id) AS num_ventas FROM ventas v JOIN trabajadores t ON
                  v.id_vendedor = t.id JOIN cargos c ON t.id_cargo = c.id
                  WHERE c.id IN (2, 3) GROUP BY t.id, t.nombre, t.apellido,
                  c.nombre_cargo ORDER BY monto_acumulado DESC LIMIT 3;"""

top_vendedores = pd.read_sql_query(q_top_vendedores, conn)
print("\nTOP 3 · Vendedores por monto acumulado")
print(top_vendedores.to_string(index=False))

conn.close()

TOP 10 · Ventas más altas (total_dte)
 id  id_cliente  id_vendedor   dte  total_dte       estado
321          75           12 10675     999825    Entregado
484          40            2 11001     998087    Entregado
628          45            2 11296     996482    Entregado
662         182            7 11360     996396    Entregado
459         144           10 10953     996276 Sin entregar
115         201            5 10253     995217    Entregado
735         110           12 11505     995031    Entregado
443         106            7 10924     994428    Entregado
179         163            3 10382     994032    Entregado
208          72           15 10440     992339    Entregado

TOP 3 · Clientes por volumen de transacciones
 id_cliente        rut   nombre apellido  cantidad_compras
        147 20416825-3  Nicolás    Silva                11
         30 11548511-3   Isabel Martínez                10
         48 19301497-6 Patricia  Herrera                10

TOP 3 · Vendedores por monto 

In [ ]:
# @title
import sqlite3
import pandas as pd
import re
# 3. Volumen: Parte tres de Análisis Exploratorio (Python)

conn = sqlite3.connect('techstore.db')
ventas = pd.read_sql_query("SELECT id, descripcion FROM ventas", conn)
conn.close()

filas = []
for _, row in ventas.iterrows():
    items = row['descripcion'].split(',')
    for item in items:
        item = item.strip()
        match = re.match(r"(\d+)x\s+(.+)", item)
        if match:
            filas.append({
                'id_venta': row['id'],
                'producto': match.group(2).strip(),
                'unidades': int(match.group(1))
            })

df_productos = pd.DataFrame(filas)

# Top 10 por unidades totales vendidas (sumando todas las ventas)
top10 = (
    df_productos
    .groupby('producto', as_index=False)
    .agg(
        unidades_totales=('unidades', 'sum'),
        num_ventas=('id_venta', 'nunique')
    )
    .sort_values('unidades_totales', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("TOP 10 · Productos más vendidos (unidades totales)")
print(top10.to_string(index=False))

TOP 10 · Productos más vendidos (unidades totales)
                         producto  unidades_totales  num_ventas
   Escáner de Documentos Portátil               551         269
     Mouse Ergonómico Inalámbrico               532         265
             Laptop Corporate 14'               504         253
                 Hub USB-C 8 en 1               502         247
            Memoria RAM 16GB DDR4               492         243
           Disco Duro Externo 2TB               492         243
Licencia Software Antivirus 1 Año               486         255
   Silla de Escritorio Ergonómica               479         242
             Teclado Mecánico RGB               475         233
    Auriculares con Micrófono USB               471         249


**Parte 3: Lógica de Negocio (Hojas de Cálculo / Excel)**

*Para esta sección, deberás ingeniártelas para extraer/descargar los datos de este entorno (ej. generando archivos CSV desde tus DataFrames o desde la DB) e importarlos a Google Sheets o Excel, puedes utilizar la tecnología que estimes conveniente, lo importante es resolver en Excel.*

1.	**Consolidación**: En una hoja nueva, toma una muestra de las primeras 100 ventas según el dte asociado (a menor folio, el documento es más antiguo). Usa fórmulas de búsqueda (BUSCARV, BUSCARX o INDICE/COINCIDIR) para traer el nombre y apellido del cliente y el nombre_cargo del vendedor asociado a esa venta.
2.	**Tabla Dinámica:** Crea una tabla dinámica que muestre el monto total vendido por cada área de la empresa (nombre_area), filtrando solo el estado "Entregado".
3.	**Segmentación:** En tu hoja de ventas, crea la columna "Categoría de Ticket". Usa funciones lógicas para clasificar la venta como "Ticket Alto" (si el total_dte supera el promedio general) o "Ticket Bajo" (si es igual o inferior).


**Opcional (Puntos extra):**

**Presentación al Negocio (Dashboard + Insights)**

1.	**Visualización**: Conecta tus datos exportados a Power BI, Looker o Tableau. Crea un dashboard de una página con:


*   KPI de Ingresos Totales y Ticket Promedio.
*   Gráfico comparativo de rendimiento por categoría/producto.
*   Filtros interactivos.
2.	**Estrategia**: En un documento breve o en celdas de texto al final de este cuaderno, responde:

*   ¿Cuáles fueron tus hallazgos respecto a la fluctuación de ingresos?
*   Propón dos recomendaciones accionables para marketing/ventas.

**Control de versiones (Github):**

1.	**Creación del repositorio:** Entregar los scripts (SQL/Python) subidos a un repositorio propio de GitHub.


Hallazgos y Recomendaciones — TechStore
Calidad de datos

Se detectaron dos problemas en la base de datos que vale la pena corregir antes de escalar cualquier solución:

Primero, existen clientes duplicados por RUT — Javiera Silva y Diego Soto aparecen cada uno con dos ID distintos en la tabla clientes. Esto fragmenta su historial de compras y distorsiona métricas como gasto histórico y retención. El RUT debería usarse como clave única al momento del registro.

Segundo, los productos no están normalizados — están almacenados como texto libre dentro de descripcion (ej. "2x Mouse, 3x Laptop..."), lo que obliga a procesamiento de texto para cualquier análisis. Lo ideal es tener una tabla productos y una tabla detalle_ventas que relacione cada venta con sus productos y cantidades.

Fluctuación de ingresos

El negocio está vendiendo bien — el problema no es la demanda sino la conversión. El 81% de las ventas están entregadas 477M de 589M, pero hay 111M en órdenes sin entregar que aún no se traducen en ingreso real. Esa brecha es el principal factor de fluctuación.

En retención, solo el 27.46% de los clientes vuelve a comprar dentro de los 200 DTE siguientes — la mayoría compra una sola vez y no regresa. Esto es costoso porque adquirir clientes nuevos siempre sale más caro que retener los existentes.

En productos, el volumen lo lideran accesorios de bajo precio (Mouse, Escáner), mientras que los productos de alto valor (Laptop, Monitor) aparecen poco en volumen pero concentran el ingreso en pocas transacciones. Eso hace al negocio vulnerable a fluctuaciones cuando esas ventas grandes no se cierran.

Recomendaciones

Primera: lanzar una campaña de reactivación dirigida a clientes que compraron hace más de 150 DTE y no han vuelto. Con los datos de qué compró cada uno, se puede personalizar la oferta (quien compró Mouse, recibe oferta de Teclado o Hub USB-C). Meta razonable: subir la retención del 27% al 40% en un trimestre.

Segunda: incentivar el upselling en productos de alto ticket. Los Ejecutivos de Ventas Senior ya generan el mayor volumen — si se les da un incentivo específico para vender Laptops y Monitores a clientes que hoy solo compran accesorios, el impacto en ingreso promedio por venta sería significativo sin necesidad de aumentar la base de clientes.